In [ ]:
# Python script for cleaning UAV orthomoasaic from Point Blue into polygon area
# STEP 2 reclassify 
# Copyright (C) 2025 Alexandra Strang  and Gorden Jiang

import rioxarray
import geopandas
import xarray as xr
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

In [ ]:
# read in raster file
downsampled_raster = rioxarray.open_rasterio("downsampled_raster.tif",
                                             chunks = "auto")

# raster info
print(downsampled_raster)
print(f"CRS of raster: {downsampled_raster.rio.crs}\n")

<xarray.DataArray (band: 1, y: 199, x: 208)> Size: 41kB
dask.array<open_rasterio-eee6fd1095c1a587f9fe4a9367fcd394<this-array>, shape=(1, 199, 208), dtype=uint8, chunksize=(1, 199, 208), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int32 4B 1
  * x            (x) float64 2kB 2.538e+05 2.538e+05 ... 2.564e+05 2.564e+05
  * y            (y) float64 2kB -1.343e+06 -1.343e+06 ... -1.345e+06 -1.345e+06
    spatial_ref  int32 4B 0
Attributes: (12/15)
    DataType:                  Generic
    RepresentationType:        ATHEMATIC
    STATISTICS_COVARIANCES:    12443.36535428216,12440.02451445284,12378.1640...
    STATISTICS_MAXIMUM:        254
    STATISTICS_MEAN:           68.32512288662
    STATISTICS_MEDIAN:         255.0
    ...                        ...
    STATISTICS_STDDEV:         92.641985431947
    AREA_OR_POINT:             Area
    STATISTICS_VALID_PERCENT:  44.73
    _FillValue:                255
    scale_factor:              1.0
    add_offset:                

In [3]:
# 1. Select the RGB bands (assuming they are the first three bands)
# rioxarray typically numbers bands starting from 1.
band = downsampled_raster.sel(band=[1])


In [ ]:
# 2. Create a boolean mask to identify pixels where any RGB value is 0 or 255 (no data values).
# The '.any(dim="band")' will check for the condition across the bands for each pixel.
condition_mask = ((band == 0) | (band == 255)).any(dim="band")


In [ ]:
# 3. Create a template for the single-band output raster.
# This ensures the output has the correct spatial metadata (CRS, transform, etc.).
# We use one of the original bands and drop the band dimension.
output_template = downsampled_raster.isel(band=0, drop=True)


In [ ]:
# 4. Reclassify the raster based on the condition.
# Where the condition is True (RGB is 0 or 255), set to 0, otherwise set to 1.
reclassified_raster = output_template.copy(
    data=xr.where(condition_mask, 0, 1)
)


In [ ]:
# 5. Set the desired nodata value for the output.
reclassified_raster = reclassified_raster.rio.write_nodata(0)

In [8]:
reclassified_raster

<xarray.DataArray (y: 199, x: 208)> Size: 166kB
dask.array<where, shape=(199, 208), dtype=int32, chunksize=(199, 208), chunktype=numpy.ndarray>
Coordinates:
  * x            (x) float64 2kB 2.538e+05 2.538e+05 ... 2.564e+05 2.564e+05
  * y            (y) float64 2kB -1.343e+06 -1.343e+06 ... -1.345e+06 -1.345e+06
    spatial_ref  int32 4B 0
Attributes: (12/15)
    DataType:                  Generic
    RepresentationType:        ATHEMATIC
    STATISTICS_COVARIANCES:    12443.36535428216,12440.02451445284,12378.1640...
    STATISTICS_MAXIMUM:        254
    STATISTICS_MEAN:           68.32512288662
    STATISTICS_MEDIAN:         255.0
    ...                        ...
    STATISTICS_STDDEV:         92.641985431947
    AREA_OR_POINT:             Area
    STATISTICS_VALID_PERCENT:  44.73
    _FillValue:                0
    scale_factor:              1.0
    add_offset:                0.0

In [ ]:
# reclassified_raster.rio.to_raster("reclassified_raster.tif", dtype="uint8") # Use a memory-efficient data type